# Data Validation

Before I start the analysis, I want to make sure the data is clean and ready to use.
I'll check for missing values, duplicates, correct data types, and basic row counts.

In [27]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

## Load the Data

I have 6 CSV files. Let me load all of them.

In [32]:
import pandas as pd

customers = pd.read_csv(
    r'C:\Users\Admin\OneDrive\Data Analyst Project\Subscription Churn Prediction & Risk Scoring\data\raw\customers.csv'
)
subscriptions = pd.read_csv(
    r'C:\Users\Admin\OneDrive\Data Analyst Project\Subscription Churn Prediction & Risk Scoring\data\raw\subscriptions.csv'
)
usage = pd.read_csv(
    r'C:\Users\Admin\OneDrive\Data Analyst Project\Subscription Churn Prediction & Risk Scoring\data\raw\usage.csv'
)
payments = pd.read_csv(
    r'C:\Users\Admin\OneDrive\Data Analyst Project\Subscription Churn Prediction & Risk Scoring\data\raw\payments.csv'
)
support_tickets = pd.read_csv(
    r'C:\Users\Admin\OneDrive\Data Analyst Project\Subscription Churn Prediction & Risk Scoring\data\raw\support_tickets.csv'
)
churn = pd.read_csv(
    r'C:\Users\Admin\OneDrive\Data Analyst Project\Subscription Churn Prediction & Risk Scoring\data\raw\churn.csv'
)

print('All files loaded.')


All files loaded.


## Row Count Check

Let me confirm how many rows each table has. This tells me if the data loaded correctly.

In [33]:
print(f'customers        : {len(customers):,} rows')
print(f'subscriptions    : {len(subscriptions):,} rows')
print(f'usage            : {len(usage):,} rows')
print(f'payments         : {len(payments):,} rows')
print(f'support_tickets  : {len(support_tickets):,} rows')
print(f'churn            : {len(churn):,} rows')

customers        : 20,000 rows
subscriptions    : 20,000 rows
usage            : 318,985 rows
payments         : 235,720 rows
support_tickets  : 40,923 rows
churn            : 20,000 rows


## Preview Each Table

Let me look at the first few rows so I understand what columns are available.

In [34]:
print('--- customers ---')
display(customers.head(3))

print('--- subscriptions ---')
display(subscriptions.head(3))

print('--- usage ---')
display(usage.head(3))

print('--- payments ---')
display(payments.head(3))

print('--- support_tickets ---')
display(support_tickets.head(3))

print('--- churn ---')
display(churn.head(3))

--- customers ---


,customer_id,signup_date,age_group,gender,region,acquisition_channel
0,CUST-10000,2023-04-13,35-44,Male,North America,Paid Ads
1,CUST-10001,2024-03-11,18-24,Female,Latin America,Organic Search
2,CUST-10002,2025-05-10,45-54,Male,Asia-Pacific,Organic Search


--- subscriptions ---


,subscription_id,customer_id,plan,billing_cycle,start_date,end_date,monthly_fee,discount_pct,auto_renew
0,SUB-10001,CUST-10000,Basic,Monthly,2023-04-13,2025-08-21,15.0,0,1
1,SUB-10002,CUST-10001,Basic,Annual,2024-03-11,2025-07-15,13.5,10,1
2,SUB-10003,CUST-10002,Basic,Monthly,2025-05-10,2025-09-10,15.0,0,0


--- usage ---


,activity_id,customer_id,activity_month,sessions_count,usage_hours,features_used_count,active_days
0,USG-10001,CUST-10000,2023-04,26,28.3,5,7
1,USG-10002,CUST-10000,2023-05,29,22.3,3,19
2,USG-10003,CUST-10000,2023-06,25,22.9,4,12


... (2 outputs left out)

--- support_tickets ---


,ticket_id,customer_id,ticket_date,issue_type,resolution_time_hours,satisfaction_score
0,TCK-10001,CUST-10000,2024-06-05,Billing,6.5,3.0
1,TCK-10002,CUST-10000,2023-10-19,Account Management,11.2,4.0
2,TCK-10003,CUST-10001,2024-09-03,Account Management,10.2,2.0


--- churn ---


,customer_id,churn_flag,churn_date,churn_reason
0,CUST-10000,0,2025-08-21,Payment Failure
1,CUST-10001,0,2025-07-15,Payment Failure
2,CUST-10002,0,2025-09-10,Competitor Switch


## Missing Values

I'll check each table for missing (null) values. Missing data can break models and distort analysis.

In [35]:
print('--- Missing Values in Each Table ---\n')

for name, df in [('customers', customers), ('subscriptions', subscriptions),
                  ('usage', usage), ('payments', payments),
                  ('support_tickets', support_tickets), ('churn', churn)]:
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls) == 0:
        print(f'{name}: No missing values')
    else:
        print(f'{name}:')
        print(nulls.to_string())
    print()

--- Missing Values in Each Table ---

customers: No missing values

subscriptions: No missing values

usage: No missing values

payments: No missing values

support_tickets:
satisfaction_score    790

churn: No missing values



**What I found:**
- `satisfaction_score` in `support_tickets` has ~790 nulls (~1.9%). This is normal — not every customer rates their ticket.
- All other columns are complete. No action needed.

## Duplicate Check

Duplicates would inflate counts and distort rates. I'll check the main ID columns.

In [36]:
print('Duplicate rows per table:')
print(f'customers       : {customers.duplicated().sum()}')
print(f'subscriptions   : {subscriptions.duplicated().sum()}')
print(f'usage           : {usage.duplicated().sum()}')
print(f'payments        : {payments.duplicated().sum()}')
print(f'support_tickets : {support_tickets.duplicated().sum()}')
print(f'churn           : {churn.duplicated().sum()}')

print('\nDuplicate customer_ids:')
print(f'customers : {customers["customer_id"].duplicated().sum()}')
print(f'churn     : {churn["customer_id"].duplicated().sum()}')

Duplicate rows per table:
customers       : 0
subscriptions   : 0
usage           : 0
payments        : 0
support_tickets : 0
churn           : 0

Duplicate customer_ids:
customers : 0
churn     : 0


## Check That All Customers Match (Referential Integrity)

Every customer_id in subscriptions, payments, etc., should exist in the main customers table.

In [37]:
valid_ids = set(customers['customer_id'])

print('Orphan records (customer_id not in customers table):')
print(f'subscriptions   : {(~subscriptions["customer_id"].isin(valid_ids)).sum()}')
print(f'usage           : {(~usage["customer_id"].isin(valid_ids)).sum()}')
print(f'payments        : {(~payments["customer_id"].isin(valid_ids)).sum()}')
print(f'support_tickets : {(~support_tickets["customer_id"].isin(valid_ids)).sum()}')
print(f'churn           : {(~churn["customer_id"].isin(valid_ids)).sum()}')

Orphan records (customer_id not in customers table):
subscriptions   : 0
usage           : 0
payments        : 0
support_tickets : 0
churn           : 0


## Convert Date Columns

Date columns are loaded as text. I need to convert them to `datetime` so I can do date math later.

In [38]:
customers['signup_date']           = pd.to_datetime(customers['signup_date'])
subscriptions['start_date']        = pd.to_datetime(subscriptions['start_date'])
subscriptions['end_date']          = pd.to_datetime(subscriptions['end_date'])
payments['payment_date']           = pd.to_datetime(payments['payment_date'])
support_tickets['ticket_date']     = pd.to_datetime(support_tickets['ticket_date'])
churn['churn_date']                = pd.to_datetime(churn['churn_date'])

print('Date columns converted successfully.')

Date columns converted successfully.


## Basic Logic Checks

A few sanity checks to confirm the data makes business sense.

In [39]:
# 1. Subscription end_date should never be before start_date
invalid = (subscriptions['end_date'] < subscriptions['start_date']).sum()
print(f'Subscriptions where end_date < start_date : {invalid}')

# 2. Payment dates should not be before customer signup
merged = payments[['customer_id', 'payment_date']].merge(customers[['customer_id', 'signup_date']], on='customer_id')
early_payments = (merged['payment_date'] < merged['signup_date']).sum()
print(f'Payments that happened before signup      : {early_payments}')

# 3. Churn rate - quick sanity check
churn_rate = churn['churn_flag'].mean() * 100
print(f'Overall churn rate                        : {churn_rate:.2f}%')

Subscriptions where end_date < start_date : 0
Payments that happened before signup      : 0
Overall churn rate                        : 33.02%


## Column Data Types

Let me confirm all columns have the right data type after conversion.

In [40]:
print('--- customers ---')
print(customers.dtypes)
print('\n--- subscriptions ---')
print(subscriptions.dtypes)

--- customers ---
customer_id                    object
signup_date            datetime64[ns]
age_group                      object
gender                         object
region                         object
acquisition_channel            object
dtype: object

--- subscriptions ---
subscription_id            object
customer_id                object
plan                       object
billing_cycle              object
start_date         datetime64[ns]
end_date           datetime64[ns]
monthly_fee               float64
discount_pct                int64
auto_renew                  int64
dtype: object


## Numerical Summary

A quick stats summary for the numeric columns to spot anything unusual.

In [41]:
print('--- Subscriptions Numeric Summary ---')
display(subscriptions[['monthly_fee', 'discount_pct']].describe().round(2))

print('--- Usage Numeric Summary ---')
display(usage[['sessions_count', 'usage_hours', 'active_days', 'features_used_count']].describe().round(2))

--- Subscriptions Numeric Summary ---


,monthly_fee,discount_pct
count,20000.00,20000.00
mean,43.65,3.95
std,51.61,6.60
min,12.00,0.00
25%,15.00,0.00
50%,15.00,0.00
75%,49.00,10.00
max,199.00,20.00


--- Usage Numeric Summary ---


,sessions_count,usage_hours,active_days,features_used_count
count,318985.00,318985.00,318985.00,318985.00
mean,27.46,24.94,13.97,4.99
std,7.68,7.83,4.54,2.54
min,3.00,0.50,1.00,1.00
25%,22.00,19.20,11.00,3.00
50%,27.00,24.90,14.00,5.00
75%,33.00,30.60,17.00,7.00
max,69.00,52.60,28.00,12.00


## Summary

Everything looks good:
- 20,000 customers, all tables fully loaded
- Zero duplicates in any table
- Zero orphan records — all foreign keys are valid
- Only expected null values (CSAT non-response ~1.9%)
- All date logic is chronologically correct
- Monthly fees range from $12 to $199, usage hours are reasonable

**Data is clean — ready for EDA!**